# 03 离散数据形式的数值微分

函数型微分可以主动选择步长；离散数据微分只能使用已有节点。若节点不等距或数据含噪声，不能简单套用等距中心差分。本节用局部插值多项式统一解释三点、五点和非等距差分权重。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import differentiate_discrete, finite_difference_weights, observed_order


## 等距数据上的三点公式

对于等距节点 $x_i=x_0+ih$，内部节点常用

$$
f'(x_i)\approx\frac{y_{i+1}-y_{i-1}}{2h}.
$$

左端点可用三点端点公式：

$$
f'(x_0)\approx\frac{-3y_0+4y_1-y_2}{2h}.
$$

这说明同一组数据在内部和边界处需要不同模板。


In [ ]:
def teaching_uniform_first_derivative(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    h = x[1] - x[0]
    d = np.empty_like(y)
    d[0] = (-3 * y[0] + 4 * y[1] - y[2]) / (2 * h)
    d[-1] = (3 * y[-1] - 4 * y[-2] + y[-3]) / (2 * h)
    d[1:-1] = (y[2:] - y[:-2]) / (2 * h)
    return d

x = np.linspace(0.0, 2.0 * math.pi, 21)
y = np.sin(x)
error = np.max(np.abs(teaching_uniform_first_derivative(x, y) - np.cos(x)))
print("最大误差：", error)


## 非等距节点：局部插值权重

给定局部节点 $x_j$，希望

$$
f^{(m)}(\xi)\approx\sum_j w_j f(x_j).
$$

要求该公式对低次多项式精确，可以得到权重方程

$$
\sum_j w_j(x_j-\xi)^k=\begin{cases}m!,&k=m,\\0,&k\ne m.\end{cases}
$$

这正是第二章局部插值思想在导数上的应用。


In [ ]:
def teaching_weights(nodes, x0, derivative_order=1):
    nodes = np.asarray(nodes, dtype=float)
    shifted = nodes - x0
    matrix = np.vander(shifted, N=nodes.size, increasing=True).T
    rhs = np.zeros(nodes.size)
    rhs[derivative_order] = math.factorial(derivative_order)
    return np.linalg.solve(matrix, rhs)

nodes = np.array([0.0, 0.3, 0.9])
weights = teaching_weights(nodes, x0=0.3, derivative_order=1)
print("教学权重：", weights)
print("正式权重：", finite_difference_weights(nodes, 0.3, 1))
print("权重和应为 0：", weights.sum())


## 非等距数据实验

下面用一个四次多项式做检查。五点局部模板对四次以下多项式的一阶导数应当精确到浮点误差量级。


In [ ]:
x = np.array([-1.0, -0.45, 0.0, 0.4, 0.95, 1.6])
y = x**4 - x**2 + 2 * x
exact = 4 * x**3 - 2 * x + 2
estimated = differentiate_discrete(x, y, derivative_order=1, stencil_size=5)
print(np.column_stack([x, estimated, exact, estimated - exact]))


## 噪声放大

若观测数据为

$$
y_i=f(x_i)+\eta_i,
$$

中心差分中的噪声项大致为

$$
\frac{\eta_{i+1}-\eta_{i-1}}{2h}.
$$

当 $h$ 较小时，即使 $\eta_i$ 很小，导数估计也可能明显振荡。


In [ ]:
rng = np.random.default_rng(2026)
x = np.linspace(0.0, 2.0 * math.pi, 81)
clean = np.sin(x)
noisy = clean + 0.02 * rng.normal(size=x.size)
exact = np.cos(x)
clean_derivative = differentiate_discrete(x, clean, stencil_size=5)
noisy_derivative = differentiate_discrete(x, noisy, stencil_size=5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x, clean, label="无噪声")
axes[0].scatter(x, noisy, s=12, label="含噪声", alpha=0.7)
axes[0].set_title("原始数据")
axes[0].legend()
axes[1].plot(x, exact, label="解析导数")
axes[1].plot(x, clean_derivative, "--", label="无噪声差分")
axes[1].plot(x, noisy_derivative, ":", label="含噪声差分")
axes[1].set_title("导数估计中的噪声放大")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.show()


## 三点与五点模板的权衡

五点模板在光滑无噪声数据上通常更精确，但它使用更宽的邻域。若数据存在噪声或局部不光滑，更宽模板不一定更好。实际应用中常需要把差分与平滑、正则化或样条方法结合起来。


In [ ]:
noise_levels = [0.0, 0.005, 0.02, 0.05]
for sigma in noise_levels:
    y_noisy = np.sin(x) + sigma * rng.normal(size=x.size)
    d3 = differentiate_discrete(x, y_noisy, stencil_size=3)
    d5 = differentiate_discrete(x, y_noisy, stencil_size=5)
    e3 = np.mean(np.abs(d3 - exact))
    e5 = np.mean(np.abs(d5 - exact))
    print(f"sigma={sigma:5.3f}  三点平均误差={e3:.3e}  五点平均误差={e5:.3e}")


## 小结

离散数据微分的关键限制是节点和数据质量已经给定。等距公式只是局部插值权重的特例；非等距节点可以通过局部权重统一处理。含噪声数据上，差分公式应谨慎使用，必要时应先做平滑或使用带正则化的模型。
